## A.I. Assignment 5

## Learning Goals

By the end of this lab, you should be able to:
* Get more familiar with tensors in pytorch 
* Create a simple multilayer perceptron model with pytorch
* Visualise the parameters


### Task

Build a fully connected feed forward network that adds two bits. Determine the a propper achitecture for this network (what database you use for this problem? how many layers? how many neurons on each layer? what is the activation function? what is the loss function? etc)

Create at least 3 such networks and compare their performance (how accurate they are?, how farst they are trained to get at 1 accuracy?)

Display for the best one the weights for each layer.


In [10]:
import torch
import torch.nn as nn

In [11]:
model1 = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 2),
    nn.Sigmoid()
)

model2 = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 2),
    nn.Sigmoid()
)

model3 = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 4),
    nn.Sigmoid(),
    nn.Linear(4, 2),
    nn.Sigmoid()
)

models = {
    "model1": model1,
    "model2": model2,
    "model3": model3,
}

model = model1

In [3]:
print(model)

Sequential(
  (fc1): Linear(in_features=2, out_features=2, bias=True)
  (act1): Sigmoid()
  (out): Linear(in_features=2, out_features=2, bias=True)
  (act_out): Sigmoid()
)


In [4]:
data_in = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
print(data_in)

tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])


In [5]:
data_target = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0],
])
print(data_target)

tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])


In [12]:
criterion = nn.BCELoss()
lr = 0.5
max_epochs = 10000

In [13]:
def train_one_model(model):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for epoch in range(1, max_epochs + 1):
        out = model(data_in)
        loss = criterion(out, data_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            pred = (out > 0.5).float()
            acc = (pred.eq(data_target).all(dim=1).float().mean()).item()

        if acc == 1.0:
            break

    return {
        "acc": acc,
        "epochs_to_1": epoch,
    }

results = {}
for name, m in models.items():
    results[name] = train_one_model(m)

print(results)

{'model1': {'acc': 1.0, 'epochs_to_1': 931}, 'model2': {'acc': 1.0, 'epochs_to_1': 792}, 'model3': {'acc': 1.0, 'epochs_to_1': 1920}}


In [14]:
for name, m in models.items():
    with torch.no_grad():
        pred = (m(data_in) > 0.5).float()
    print(name, "pred:")
    print(pred)
    print("target:")
    print(data_target)
    print("stats:", results[name])
    print("-")

model1 pred:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
target:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
stats: {'acc': 1.0, 'epochs_to_1': 931}
-
model2 pred:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
target:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
stats: {'acc': 1.0, 'epochs_to_1': 792}
-
model3 pred:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
target:
tensor([[0., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])
stats: {'acc': 1.0, 'epochs_to_1': 1920}
-


In [15]:
best_model_name = min(results, key=lambda k: results[k]["epochs_to_1"])
best_model = models[best_model_name]

print("best model:", best_model_name)
print("weights:")
for name, p in best_model.named_parameters():
    print(name)
    print(p.data)

best model: model2
weights:
0.weight
tensor([[ 0.8719,  0.9704],
        [ 2.0277,  2.2764],
        [-3.1124, -2.9534],
        [-0.7349, -1.2804]])
0.bias
tensor([-1.1371, -1.0477,  4.6098,  0.3880])
2.weight
tensor([[ 1.7989,  3.3903, -6.4891, -1.4675],
        [-0.7744,  1.3406,  1.7320, -0.8085]])
2.bias
tensor([-0.5694, -1.3959])
